In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, RepeatVector, TimeDistributed, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import os # Import the os module for path manipulation

# Download and load the real dataset
zip_path = tf.keras.utils.get_file(
    origin='https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname='jena_climate_2009_2016.csv.zip',
    extract=True
)

# Construct the full path to the CSV file
csv_file_path = os.path.join(zip_path, 'jena_climate_2009_2016.csv')
df = pd.read_csv(csv_file_path)

# Extract features: Temperature (degC), Pressure (mbar), Relative Humidity (%)
features = df[['T (degC)', 'p (mbar)', 'rh (%)']].values[:1000]
temp_univariate = features[:, 0]

# Helper function to structure time-series sequences
def create_sequences(data, time_steps=20):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i + time_steps])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

# Prepare 1D Univariate Data (20 time steps -> 1 feature)
X_uni, y_uni = create_sequences(temp_univariate, time_steps=20)
X_uni = X_uni.reshape((X_uni.shape[0], X_uni.shape[1], 1))

# Prepare 3D Multivariate Data (20 time steps -> 3 features)
X_multi, y_multi = create_sequences(features, time_steps=20)
y_multi = y_multi[:, 0] # Target: Temperature

# Prepare Many-to-Many target sequences (20 time steps output)
y_seq = np.array([temp_univariate[i+1 : i+21] for i in range(len(temp_univariate) - 20)])
y_seq = y_seq.reshape((y_seq.shape[0], y_seq.shape[1], 1))

In [2]:
model_vanilla = Sequential([
    LSTM(50, input_shape=(20, 1)),
    Dense(1)
])
model_vanilla.compile(optimizer="adam", loss="mse")
model_vanilla.summary()
model_vanilla.fit(X_uni, y_uni, epochs=50, batch_size=32, verbose=1)

# Predict next value
sample_input = X_uni[0].reshape((1, 20, 1))
prediction = model_vanilla.predict(sample_input, verbose=0)
print("Predicted next temperature:", prediction[0][0])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 75.1027
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 45.1621
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 24.5827
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 15.3768
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 10.4171
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.3482
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 5.4531
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.2345
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 3.3205
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.6398
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.1171
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.7092
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4005
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1284
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9204
Epoch 16/50
31

In [3]:
model_stacked = Sequential([
    LSTM(64, return_sequences=True, input_shape=(20, 1)),
    LSTM(32),
    Dense(1)
])
model_stacked.compile(optimizer="adam", loss="mse")
model_stacked.summary()
model_stacked.fit(X_uni, y_uni, epochs=50, batch_size=32, verbose=1)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 20, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 61.9764
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 31.1468
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 20.9935
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 16.9061
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 13.8856
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 11.5691
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9.6944
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 8.1658
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 6.9304
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.8887
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.0502
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.3869
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.7841
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.2667
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.8697
Epoch 16/5

In [4]:
model_multi = Sequential([
    LSTM(64, input_shape=(20, 3)),
    Dense(1)
])
model_multi.compile(optimizer="adam", loss="mse")
model_multi.summary()
model_multi.fit(X_multi, y_multi, epochs=50, batch_size=32, verbose=1)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,473 (68.25 KB)

 Trainable params: 17,473 (68.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 77.6073
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 66.4174
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 57.4980
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 52.3536
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 48.2991
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 45.2749
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 42.9529
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 41.2783
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 40.0799
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 39.2302
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 38.6002
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 38.2387
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 38.0101
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.8678
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.7627
Epoc

In [5]:
model_m2o = Sequential([
    LSTM(50, input_shape=(20, 1)),
    Dense(1)
])
model_m2o.compile(optimizer="adam", loss="mse")
model_m2o.summary()
model_m2o.fit(X_uni, y_uni, epochs=50, batch_size=32, verbose=1)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 72.5168
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 38.6249
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 21.5457
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14.4221
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.2336
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7.4403
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5.5896
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 4.3756
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.3270
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.5280
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.9592
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.5021
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1677
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.9100
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7375
Epoch 16/50
31

In [6]:
model_m2m = Sequential([
    LSTM(64, return_sequences=True, input_shape=(20, 1)),
    TimeDistributed(Dense(1))
])
model_m2m.compile(optimizer="adam", loss="mse")
model_m2m.summary()
model_m2m.fit(X_uni, y_seq, epochs=50, batch_size=32, verbose=1)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 20, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 1)          │            65 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 73.0182
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 39.4762
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 24.2453
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 16.3839
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 11.5174
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 8.0815
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 5.9132
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 4.4263
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.3788
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.6682
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.1479
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.7655
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.4673
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.2432
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.0597
Epoch 16/50
31

In [7]:
model_autoencoder = Sequential([
    # Encoder
    LSTM(64, input_shape=(20, 1)),
    # Replicate bottleneck vector across 20 time steps
    RepeatVector(20),
    # Decoder
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(1))
])
model_autoencoder.compile(optimizer="adam", loss="mse")
model_autoencoder.summary()
# Learns to reconstruct its input sequence
model_autoencoder.fit(X_uni, X_uni, epochs=50, batch_size=32, verbose=1)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 20, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 1)          │            65 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,985 (195.25 KB)

 Trainable params: 49,985 (195.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 58.4802
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 19.7387
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.8570 
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.2925
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.3768
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.2085
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.4300
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.8828
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.5076
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2236
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.0097
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.8733
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.7747
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6933
Epoch 15/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.6901
Epoch 16/50
3

In [8]:
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Define Option 1: Small text corpus (Climate/Weather domain)
sentences = [
    "temperature and pressure affect daily weather conditions",
    "lstm neural networks learn sequential time series patterns",
    "deep learning predicts multi variable sensor parameters accurately",
    "weather forecasting uses historical sequence data for accuracy"
]

# 2. Tokenize text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1  # +1 for 0-padding index

# 3. Generate N-gram input sequences and target next-words
input_sequences = []
for line in sentences:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 4. Pad sequences so all inputs have equal length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Separate features (X: all words except last) and labels (y: last word in sequence)
X_text = input_sequences[:, :-1]
y_text = input_sequences[:, -1]

# 5. Build Text / Next-Word LSTM Model
model_text = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16), # Removed input_length
    LSTM(64),
    Dense(vocab_size, activation="softmax") # Softmax outputs probability across vocabulary
])

# Compile the model before training
model_text.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_text.summary()

# 6. Train the model
model_text.fit(X_text, y_text, epochs=100, verbose=0)

# 7. Predict the next word given a seed text prompt
seed_text = "weather forecasting"
token_list = tokenizer.texts_to_sequences([seed_text])[0]
padded_seed = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

# Get highest probability index
predicted_probs = model_text.predict(padded_seed, verbose=0)
predicted_index = np.argmax(predicted_probs, axis=-1)[0]

# Convert predicted index back to word
predicted_word = ""
for word, index in tokenizer.word_index.items():
    if index == predicted_index:
        predicted_word = word
        break

print(f"Seed Prompt: '{seed_text}'")
print(f"Predicted Next Word: '{predicted_word}'")

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Seed Prompt: 'weather forecasting'
Predicted Next Word: 'forecasting'


In [9]:
model_dropout = Sequential([
    LSTM(64, input_shape=(20, 1)),
    Dropout(0.2), # Randomly zeroes out 20% of activations during training
    Dense(1)
])
model_dropout.compile(optimizer="adam", loss="mse")
model_dropout.summary()
model_dropout.fit(X_uni, y_uni, epochs=500, batch_size=32, verbose=1)

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_9 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 62.8495
Epoch 2/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 29.8990
Epoch 3/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 15.6317
Epoch 4/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.1570 
Epoch 5/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 5.7244
Epoch 6/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 4.1156
Epoch 7/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.1646
Epoch 8/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.6297
Epoch 9/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.1485
Epoch 10/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4809
Epoch 11/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4273
Epoch 12/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.2199
Epoch 13/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.9968
Epoch 14/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.9858
Epoch 15/500
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.8863
